### 处理数据用于100-shot Aug实验

In [1]:
# from utils import extract_entities, save_json_as_csv, load_json, get_ners, save_json
# import os
# import random
# import numpy as np

# # Configuration
# domain = "music"  # Change this to the desired domain
# input_path = f"./datasets/aug/zero_aug/{domain}.json"
# output_dir = f"./datasets/aug/{domain}"

# random_seed = 42

# # Load data
# data = load_json(input_path)


# seed_indices = []
# for i in range(len(data)):
#     if 'train' in data[i]['sent_id']:
#         seed_indices.append(i)

# # Split into seed and remaining (unlabeled) data
# seed_data = [data[i] for i in seed_indices]
# rest_data = [data[i] for i in range(len(data)) if i not in seed_indices]

# print(f"Procssing {domain}...")
# print(f"Number of samples in seed data: {len(seed_data)}")
# print(f"Number of samples in rest data: {len(rest_data)}")

# # Ensure output directory exists
# os.makedirs(output_dir, exist_ok=True)

# # randomly sample 100 samples from seed_data
# if len(seed_data) > 100:
#     print(f"There are {len(seed_data)} samples in seed data")
#     print("Randomly sampling 100 samples from seed data")
#     random.seed(random_seed)
#     sub_seed = random.sample(seed_data, 100)
#     # Save the splits
#     save_json(seed_data, os.path.join(output_dir, "200_seed.json"))
#     save_json(sub_seed, os.path.join(output_dir, "seed.json"))
# else:
#     save_json(seed_data, os.path.join(output_dir, "seed.json"))
# save_json(rest_data, os.path.join(output_dir, f"unlabel.json"))

## 新思路（利用seed数据构建judge和modifer的demos）


In [4]:
from typing import List, Dict, Set, Tuple
import json # For example printing

def generate_error_demos(
    llm_entities_list: List[List[str]],
    gold_entities_list: List[List[str]]
) -> Dict[str, list]:
    """
    Compares LLM annotation list to Gold standard list (without offsets)
    to identify specific instances of Type, Span, Missing, and Spurious errors
    for generating few-shot examples.

    Args:
        llm_entities_list: List of [text, type] from LLM annotation.
                           e.g., [['perceptron', 'algorithm'], ['Perceptrons', 'product']]
        gold_entities_list: List of [text, type] from Gold standard.
                            e.g., [['perceptron models', 'algorithm'], ['Perceptrons', 'miscellaneous']]

    Returns:
        A dictionary containing lists of identified errors:
        {
            'type': [ [llm_text, llm_type, gold_text, gold_type], ... ],
            'span': [ [llm_text, llm_type, gold_text, gold_type], ... ],
            'missing': [ [gold_text, gold_type], ... ],
            'spurious': [ [llm_text, llm_type], ... ]
        }
    """
    # --- 1. Pre-process Inputs ---
    # Use tuples {(text, type)} for efficient set operations
    llm_set: Set[Tuple[str, str]] = {tuple(item) for item in llm_entities_list}
    gold_set: Set[Tuple[str, str]] = {tuple(item) for item in gold_entities_list}

    # Create maps for text -> type lookup.
    # NOTE: This simple dict assumes unique text strings. If the same text can appear
    # with different types *within the same list*, this might need adjustment.
    # For this specific task comparing LLM vs Gold, it's usually sufficient.
    llm_map: Dict[str, str] = {text: type_ for text, type_ in llm_entities_list}
    gold_map: Dict[str, str] = {text: type_ for text, type_ in gold_entities_list}

    # --- 2. Initialize Results & Tracking ---
    type_errors: List[List[str]] = []
    span_errors: List[List[str]] = []
    missing: List[List[str]] = []
    spurious: List[List[str]] = []

    # Keep track of tuples that have been definitively matched (Exact, Type, Span)
    # This prevents double counting errors or misclassifying parts of pairs.
    matched_llm_tuples: Set[Tuple[str, str]] = set()
    matched_gold_tuples: Set[Tuple[str, str]] = set()

    # --- 3. Pass 1: Exact Matches (Text + Type) ---
    # Find entities present in both sets
    exact_matches = llm_set.intersection(gold_set)
    matched_llm_tuples.update(exact_matches)
    matched_gold_tuples.update(exact_matches)
    # print(f"Exact Matches: {exact_matches}") # Debugging

    # --- 4. Pass 2: Type Errors (Same Text, Different Type) ---
    # Iterate through LLM entities not already exactly matched
    for llm_text, llm_type in llm_set:
        llm_tuple = (llm_text, llm_type)
        if llm_tuple in matched_llm_tuples:
            continue

        # Check if the same text exists in gold standard with a different type
        if llm_text in gold_map:
            gold_type = gold_map[llm_text]
            if llm_type != gold_type:
                gold_tuple = (llm_text, gold_type)
                # Ensure this corresponding gold entity actually exists in the gold set
                # and hasn't already been matched (e.g., exactly matched with a different LLM entity if duplicates existed)
                if gold_tuple in gold_set and gold_tuple not in matched_gold_tuples:
                    type_errors.append([llm_text, llm_type, llm_text, gold_type])
                    matched_llm_tuples.add(llm_tuple)
                    matched_gold_tuples.add(gold_tuple)
                    # print(f"Type Error Found: {llm_tuple} vs {gold_tuple}") # Debugging


    # --- 5. Pass 3: Span Errors (Substring Overlap Heuristic) ---
    # Iterate through remaining unmatched LLM entities
    for llm_text, llm_type in llm_set:
        llm_tuple = (llm_text, llm_type)
        if llm_tuple in matched_llm_tuples:
            continue

        # Compare against remaining unmatched Gold entities
        for gold_text, gold_type in gold_set:
            gold_tuple = (gold_text, gold_type)
            if gold_tuple in matched_gold_tuples:
                continue

            # Check for non-identical text AND substring relationship
            # Ensure texts are not empty before checking 'in'
            if llm_text and gold_text and llm_text != gold_text and \
               (llm_text in gold_text or gold_text in llm_text):
                # Found a potential span error match
                span_errors.append([llm_text, llm_type, gold_text, gold_type])
                matched_llm_tuples.add(llm_tuple)
                matched_gold_tuples.add(gold_tuple)
                # print(f"Span Error Found: {llm_tuple} vs {gold_tuple}") # Debugging
                # Break inner loop: Match this LLM entity to the first overlapping Gold entity found
                # This prevents one LLM entity matching multiple Gold entities as span errors.
                break

    # --- 6. Pass 4: Missing and Spurious ---
    # Missing: Gold entities not matched in any previous pass
    for gold_text, gold_type in gold_set:
        gold_tuple = (gold_text, gold_type)
        if gold_tuple not in matched_gold_tuples:
            missing.append([gold_text, gold_type])
            # print(f"Missing Found: {gold_tuple}") # Debugging


    # Spurious: LLM entities not matched in any previous pass
    for llm_text, llm_type in llm_set:
        llm_tuple = (llm_text, llm_type)
        if llm_tuple not in matched_llm_tuples:
            spurious.append([llm_text, llm_type])
            # print(f"Spurious Found: {llm_tuple}") # Debugging


    # --- 7. Return Result Dictionary ---
    return {
        'type': type_errors,
        'span': span_errors,
        'missing': missing,
        'spurious': spurious
    }

In [3]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

for i in [5, 25, 50, 100, 200, 300, 400, 500]:
    # data_path = f"./output/conll2003/deepseek-chat_BM25_{i}.json"
    data_path = f"./output/conll2003/deepseek-chat_BM25_{i}.json"
    data = load_json(data_path)
    counts = Counter()
    for sent in data:
        if "response" not in sent:
            continue
        sent['response'] = sent['response'].replace("Output: ", "")
        pred = extract_entities(sent['response'])
        ref = sent['entities']
        counts = evaluate_sent(ref, pred, counts)
    scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
    # print(f"======= {i} =======")
    print(scores_ner)

{'precision': 0.808, 'recall': 0.8744588744588745, 'f1': 0.8399168399168399}
{'precision': 0.8789808917197452, 'recall': 0.8961038961038961, 'f1': 0.8874598070739549}
{'precision': 0.8891257995735607, 'recall': 0.9025974025974026, 'f1': 0.8958109559613319}
{'precision': 0.8919491525423728, 'recall': 0.9112554112554112, 'f1': 0.9014989293361884}
{'precision': 0.9006342494714588, 'recall': 0.922077922077922, 'f1': 0.9112299465240641}
{'precision': 0.8900634249471459, 'recall': 0.9112554112554112, 'f1': 0.9005347593582886}
{'precision': 0.8907563025210085, 'recall': 0.9177489177489178, 'f1': 0.9040511727078893}
{'precision': 0.893305439330544, 'recall': 0.9242424242424242, 'f1': 0.9085106382978724}


**对Demo数据进行统计和构建prompt**

In [6]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "output/aug/literature/deepseek-chat_50-ann4reflection.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['entities']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['entities']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

{'precision': 0.8210526315789474, 'recall': 0.850909090909091, 'f1': 0.8357142857142856}
Total: 50, No errors: 17, Errors: 33
Errors statistics:
type: 24
span: 15
missing: 2
spurious: 11


In [8]:
# save 
save_json(data, "./datasets/aug/literature/refiner-demos-50-errors.json")

## 处理dev set数据，验证我们的error-aware refinment方法

In [9]:
# 
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/Qwen2.5-72B-demo100-ann-dev-logits.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")
# save as json 
# save_json(data, "./output/aug/ai/deepseek-chat-demo100-ann-dev-logits-errors.json")

{'precision': 0.7357414448669202, 'recall': 0.749515816655907, 'f1': 0.7425647585545252}
Total: 350, No errors: 134, Errors: 216
Errors statistics:
type: 183
span: 98
missing: 102
spurious: 129


In [3]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/deepseek-chat-demo100-ann-dev-logits.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")
# save as json 
# save_json(data, "./output/aug/ai/deepseek-chat-demo100-ann-dev-logits-errors.json")

{'precision': 0.7344527363184079, 'recall': 0.7624273724983861, 'f1': 0.748178650617675}
Total: 350, No errors: 128, Errors: 222
Errors statistics:
type: 195
span: 97
missing: 67
spurious: 127


In [21]:
# output/aug/ai/Qwen2.5-72B-demo100-ann-dev-logits.json

import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/refiner/Qwen2.5-72B-pos100-neg0-refiner-all-new.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")
# save as json 
# save_json(data, "./output/aug/ai/Qwen2.5-72B-demo100-ann-dev-logits-errors.json")

{'precision': 0.7119952494061758, 'recall': 0.7536140791954745, 'f1': 0.7322137404580153}
Total: 350, No errors: 111, Errors: 239
Errors statistics:
type: 186
span: 116
missing: 84
spurious: 175


### Spurious Errors

**构建用于Spurious examples**

**evaluate** Spurious 效果

In [3]:
import ast
import re
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

def clean_llm_output(text):
    """清理输入字符串：移除首尾空白和常见的Markdown代码块标记。"""
    if not isinstance(text, str):
        return ""
    # 首先移除首尾空白
    text = text.strip()

    # 处理 ``` 代码块标记 (尝试更可靠地移除)
    # 情况 1: 包裹形式 ```[lang]\n...\n```
    if text.startswith('```') and text.endswith('```'):
        # 移除外层 ```
        text = text[3:-3].strip()
        # 尝试移除可选的语言标记行 (简单判断：如果第一行不像数据且只有单个词)
        if '\n' in text:
            first_line, rest = text.split('\n', 1)
            stripped_first_line = first_line.strip()
            if stripped_first_line and not stripped_first_line.startswith(('[', '{', "'", '"')) and len(stripped_first_line.split()) <= 1:
                text = rest.strip()

    # 情况 2: 只有结尾标记，如 "[]\n```"
    elif text.endswith("\n```"):
        text = text[:-4].rstrip()

    # 再次确保移除首尾空白
    return text.strip()

def parse_llm_list_string_fix_first(llm_output_str):
    """
    优先尝试修复字符串末尾缺失的']'，然后解析整个字符串。

    Args:
        llm_output_str: LLM生成的原始字符串。

    Returns:
        如果解析成功且结果是列表，则返回该列表。
        否则返回空列表 []。
    """
    if not isinstance(llm_output_str, str):
        return []

    cleaned_str = clean_llm_output(llm_output_str)
    if not cleaned_str:
        return []

    # --- 步骤 1 & 2: 检查是否需要修复 ---
    string_to_parse = cleaned_str
    # 检查条件: 以'['开头 并且 '['数量 > ']'数量
    if cleaned_str.startswith('[') and cleaned_str.count('[') > cleaned_str.count(']'):
        # 计算需要补充多少个 ']'
        open_brackets = cleaned_str.count('[')
        close_brackets = cleaned_str.count(']')
        missing_count = open_brackets - close_brackets
        # 构造修复后的字符串
        string_to_parse = cleaned_str + ']' * missing_count
        # print(f"Debug: Fixing applied: {repr(cleaned_str)} -> {repr(string_to_parse)}") # 可选的调试信息

    # else:
        # print(f"Debug: No fixing needed for: {repr(cleaned_str)}") # 可选的调试信息


    # --- 步骤 3: 尝试解析最终的字符串 (可能是原始的，也可能是修复后的) ---
    try:
        parsed_result = ast.literal_eval(string_to_parse)
        # 检查解析结果是否为列表
        if isinstance(parsed_result, list):
            return parsed_result # 成功，返回解析出的列表
        else:
            # 解析成功但不是列表 (例如解析出字符串、数字等)
            return []
    except (SyntaxError, ValueError, TypeError, MemoryError):
        # 解析失败 (原始字符串无效，或修复后仍然无效)
        # print(f"Debug: Parsing failed for: {repr(string_to_parse)}") # 可选的调试信息
        return [] # 返回空列表

In [4]:
# before spurious
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/deepseek-chat-demo100-ann-dev-logits-errors.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7344527363184079, 'recall': 0.7624273724983861, 'f1': 0.748178650617675}


In [15]:
data_path = "./output/aug/ai/refiner/deepseek-chat-dev-filter-spurious-refiner-demo100.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if not sent['refine']:
        pred = extract_entities(sent['response'])
    else:
        # spurious_ents = parse_llm_list_string_fix_first(sent['spurious_refined_response'])
        spurious_ents = sent['spurious_refined_response']
        pred = extract_entities(sent['response'])
        # remove spurious entities from the pred list
        for spurious_ent in spurious_ents:
            # print(f"Removing spurious entity: {spurious_ent}")
            if spurious_ent in pred:
                pred.remove(spurious_ent)
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7546563904945408, 'recall': 0.7585539057456423, 'f1': 0.7566001287830008}


In [7]:
# before spurious
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/Qwen2.5-72B-demo100-ann-dev-logits-errors.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7357414448669202, 'recall': 0.749515816655907, 'f1': 0.7425647585545252}


In [ ]:
data_path = "./output/aug/ai/refiner/Qwen2.5-72B-dev-filter-spurious-refiner-demo100.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if not sent['refine']:
        pred = extract_entities(sent['response'])
    else:
        spurious_ents = parse_llm_list_string_fix_first(sent['spurious_refined_response'])
        pred = sent['pred']
        # remove spurious entities from the pred list
        for spurious_ent in spurious_ents:
            # print(f"Removing spurious entity: {spurious_ent}")
            if spurious_ent in pred:
                pred.remove(spurious_ent)
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7582128777923784, 'recall': 0.7449967721110394, 'f1': 0.751546727450342}


In [14]:
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-type-refiner-spurious-refiner-demo100.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "spurious_refined_response" not in sent:
        continue
    spurious_ents = parse_llm_list_string_fix_first(sent['spurious_refined_response'])
    pred = sent['type_refineed_pred']
    # remove spurious entities from the pred list
    for spurious_ent in spurious_ents:
        # print(f"Removing spurious entity: {spurious_ent}")
        if spurious_ent in pred:
            pred.remove(spurious_ent)
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.8, 'recall': 0.7178825048418335, 'f1': 0.7567199727798571}


In [13]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./datasets/aug/ai/refiner-demos.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    sent['pred'] = pred
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = sent['pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")
# save as json
save_json(data, "./datasets/aug/ai/refiner-demos-errors.json")

{'precision': 0.7168316831683168, 'recall': 0.7718550106609808, 'f1': 0.7433264887063655}
Total: 90, No errors: 20, Errors: 70
Errors statistics:
type: 59
span: 28
missing: 21
spurious: 52


### Missing Errors

**构建用于创建missing examples的函数**

In [8]:
# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100--ann-dev-missing-refiner-new-100.json"
data = load_json(data_path)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['missing_refined_response'])
    # pred = sent['pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

Total: 350, No errors: 126, Errors: 224
Errors statistics:
type: 191
span: 100
missing: 62
spurious: 142


In [9]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100--ann-dev-missing-refiner-new-100.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "missing_refined_response" not in sent:
        continue
    pred = extract_entities(sent['missing_refined_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7312883435582822, 'recall': 0.7695287282117496, 'f1': 0.7499213589178987}


In [15]:
# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100--ann-dev-missing-refiner-new-100.json"
data = load_json(data_path)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['missing_refined_response'])
    # pred = sent['pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

Total: 350, No errors: 126, Errors: 224
Errors statistics:
type: 191
span: 101
missing: 61
spurious: 141


In [16]:
# parse_llm_list_string_fix_first

def fix_nested_pred_list(pred_list):
    fixed_list = []
    for item in pred_list:
        # 如果是嵌套了一层，例如 [['a', 'b']]，就取里面那一层
        while isinstance(item, list) and len(item) == 1 and isinstance(item[0], list):
            item = item[0]
        # 最终确认是长度为2的扁平 list
        if isinstance(item, list) and len(item) == 2 and all(isinstance(i, str) for i in item):
            fixed_list.append(item)
        else:
            print(f"警告：跳过了格式异常的项: {item}")
    return fixed_list

import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
# data_path = "./output/aug/ai/refiner/deepseek-chat-demo100--ann-dev-missing-refiner-new-list.json"
# data_path = "output/aug/ai/refiner/Qwen2.5-72B/Qwen2.5-72B-demo100-dev-filter-missing-refiner.json" # Qwen的结果
# data_path = "output/aug/ai/refiner/deepseek-chat/deepseek-chat-demo100-dev-filter-missing-refiner.json" # DeepSeek的结果
data_path = "output/aug/ai/refiner/deepseek-chat/deepseek-chat-demo100-dev-filter-suprious-missing-refiner.json" # DeepSeek的S+M结果
data = load_json(data_path)
counts = Counter()
for sent in data:
    pred = extract_entities(sent['response'])
    if "missing_refined_response" in sent:
        missing_refined_response = sent['missing_refined_response']
        if missing_refined_response == "None" or missing_refined_response == None:
            missing_list = []
        else:
            missing_list = parse_llm_list_string_fix_first(sent['missing_refined_response'])
    else:
        missing_list = []
    # 让missing_list的维度和pred一致，都是[[text, type], ...]
    # add missing_list to pred
    for missing_ent in missing_list:
        # print(f"Adding missing entity: {missing_ent}")
        if missing_ent not in pred:
            pred.append(missing_ent)
    pred = fix_nested_pred_list(pred)
    sent['pred'] = pred
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    # missing_pred = extract_entities(sent['missing_refined_response'])
    # # add missing_list to pred
    pred = sent['pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

{'precision': 0.7389192471159685, 'recall': 0.7856681730148483, 'f1': 0.7615769712140176}
Total: 350, No errors: 131, Errors: 219
Errors statistics:
type: 191
span: 97
missing: 35
spurious: 134


In [18]:
pred

[['computer vision', 'field'],
 ['Facial recognition', 'task'],
 ['k -NN', 'algorithm'],
 ['feature extraction', 'task'],
 ['dimension reduction', 'task'],
 ['OpenCV', 'product'],
 [['pre-processing steps', 'miscellaneous']]]

## Span Error

**创建用于Span Error的数据**

In [17]:
data[1]['ners']

[['SVM', 'algorithm'],
 ['classification algorithms', 'miscellaneous'],
 ['regularized least-squares', 'algorithm'],
 ['logistic regression', 'algorithm']]

**Evaluate Span Error**

In [23]:
# 
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-span-refiner-demo50.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "span_refined_response" not in sent:
        continue
    sent['span_refined_response'] = sent['span_refined_response'].replace("Refined Annotated Text:", "")
    pred = extract_entities(sent['span_refined_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['span_refined_response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

{'precision': 0.7261687917425622, 'recall': 0.7721110393802453, 'f1': 0.7484355444305382}
Total: 350, No errors: 121, Errors: 229
Errors statistics:
type: 192
span: 122
missing: 68
spurious: 127


In [22]:

import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-simple-span-refiner-demo50.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "span_refined_response" not in sent:
        continue
    sent['span_refined_response'] = sent['span_refined_response'].replace("Refined Annotated Text:", "")
    # if start with "Initial Annotated Text:", split it to two parts by "\n"
    if sent['span_refined_response'].startswith("Initial Annotated Text:"):
        sent['span_refined_response'] = sent['span_refined_response'].split("\n", 1)[1]
    # sent['span_refined_response'] = sent['span_refined_response'].replace("Initial Annotated Text:", "")
    sent['span_refined_response'] = sent['span_refined_response'].strip()
    pred = extract_entities(sent['span_refined_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['span_refined_response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

{'precision': 0.7185554171855542, 'recall': 0.7449967721110394, 'f1': 0.7315372424722664}
Total: 350, No errors: 116, Errors: 234
Errors statistics:
type: 190
span: 123
missing: 78
spurious: 135


In [18]:
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-simple-span-refiner-demo100.json"
data = load_json(data_path)
counts = Counter()
span_preds = []
response_preds = []
refs_list = []
for sent in data:
    sent['span_refined_response'] = sent['span_refined_response'].replace("Refined Annotated Text:", "")
    sent['span_refined_response'] = sent['span_refined_response'].replace("Initial Annotated Text:", "")
    sent['span_refined_response'] = sent['span_refined_response'].strip()
    span_pred = extract_entities(sent['span_refined_response'])
    response_pred = extract_entities(sent['response'])
    span_preds.append(span_pred)
    response_preds.append(response_pred)
    ref = sent['ners']
    refs_list.append(ref)
#     counts = evaluate_sent(ref, pred, counts)
# scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# # print(f"======= {i} =======")
# print(scores_ner)

In [19]:
i = 0
print(f"Text: {data[i]['text']}")
print(f"Reponse Pred: {response_preds[i]}")
print(f"Span Pred: {span_preds[i]}")
print(f"Ref: {refs_list[i]}")

Text: Here , accuracy is measured by error rate , which is defined as :
Reponse Pred: [['accuracy', 'metrics'], ['error rate', 'metrics']]
Span Pred: [['accuracy', 'metrics'], ['error rate', 'metrics'], ['accuracy', 'metrics'], ['error rate', 'metrics']]
Ref: [['accuracy', 'metrics'], ['error rate', 'metrics']]


**refine后的数据**

In [4]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/deepseek-chat-demo100-ann-dev.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

{'precision': 0.7400497512437811, 'recall': 0.7682375726275016, 'f1': 0.753880266075388}
Total: 350, No errors: 130, Errors: 220
Errors statistics:
type: 189
span: 99
missing: 67
spurious: 126


### **Typing部分** Final

In [7]:
# output/aug/ai/refiner/deepseek-chat-demo100-retriveTrue-ann-dev-type-refinernew-definition.json

import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
# data_path = "./output/aug/ai/refiner/deepseek-chat/dev-type-filter-refiner-demo.json"
# data_path = "./output/aug/ai/refiner/Qwen2.5-72B/dev-type-filter-refiner-demo.json"
data_path = "output/aug/ai/refiner/deepseek-chat/dev-filter-spurious-missing-type-refiner.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if sent['refine'] == False:
        pred = extract_entities(sent['response'])
    else:
        sent['type_refined_response'] = sent['type_refined_response'].replace("Refined Entity List: ", "")
        # remove white spaces at beginning and end
        sent['type_refined_response'] = sent['type_refined_response'].strip()
        pred = ast.literal_eval(sent['type_refined_response'])
    # save this to a new field
    sent['type_refined_pred'] = pred
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7782002534854245, 'recall': 0.7927695287282117, 'f1': 0.7854173329069395}


In [3]:
import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100False-ann-dev-type-refinernew-definition.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    # if "response" not in sent:
    #     continue
    # sent['response'] = sent['response'].replace("Output: ", "")
    sent['type_refined_response'] = sent['type_refined_response'].replace("Refined Entity List: ", "")
    # remove white spaces at beginning and end
    sent['type_refined_response'] = sent['type_refined_response'].strip()
    pred = ast.literal_eval(sent['type_refined_response'])
    # save this to a new field
    sent['type_refined_pred'] = pred
    old_pred = sent['pred']
    assert len(pred) == len(old_pred)
    # pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7549751243781094, 'recall': 0.7837314396384765, 'f1': 0.7690845739626229}


In [3]:
# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = sent['type_refined_pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

NameError: name 'generate_error_demos' is not defined

Popular approaches of opinion-based <entity type="product">recommender system</entity> utilize various techniques including <entity type="task">text mining</entity> , <entity type="task">information retrieval</entity> , <entity type="task">sentiment analysis</entity> ( see also <entity type="task">Multimodal sentiment analysis</entity> ) and <entity type="field">deep learning</entity> <entity type="researcher">X.Y. Feng</entity> , <entity type="researcher">H. Zhang</entity> , <entity type="researcher">Y.J. Ren</entity> , <entity type="researcher">P.H. Shang</entity> , <entity type="researcher">Y. Zhu</entity> , <entity type="researcher">Y.C. Liang</entity> , <entity type="researcher">R.C. Guan</entity> , <entity type="researcher">D. Xu</entity> , ( 2019 ) , , 21 ( 5 ) : e12957 .


In [2]:
# output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-type-refiner.json

import ast
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-type-refiner.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    # if "response" not in sent:
    #     continue
    # sent['response'] = sent['response'].replace("Output: ", "")
    sent['type_refined_response'] = sent['type_refined_response'].replace("Refined Entity List: ", "")
    # remove white spaces at beginning and end
    sent['type_refined_response'] = sent['type_refined_response'].strip()
    pred = ast.literal_eval(sent['type_refined_response'])
    # save this to a new field
    sent['type_refined_pred'] = pred
    # pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7450248756218906, 'recall': 0.7734021949644933, 'f1': 0.7589483687044664}


In [9]:
# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = sent['type_refined_pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

Total: 350, No errors: 131, Errors: 219
Errors statistics:
type: 181
span: 99
missing: 67
spurious: 126


In [10]:
# save
save_json(data, "./output/aug/ai/refiner/deepseek-chat-demo100-ann-dev-type-refiner-errors.json")

In [ ]:
# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    # sent['response'] = sent['response'].replace("Output: ", "")
    pred = sent['type_refined_pred']
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")

Total: 350, No errors: 130, Errors: 220
Errors statistics:
type: 189
span: 99
missing: 67
spurious: 126


In [19]:
# save to json 
save_json(data, "./datasets/aug/ai/deepseek-chat-0-shots-ICL-errors.json")

In [ ]:
# randomly select 5 samples and print them to check the correctness of generate_error_demos function
# random.seed(42)
random_samples = random.sample(data, 5)
for sample in random_samples:
    print(f"Text: {sample['text']}")
    print(f"Annotated Text: {sample['response']}")
    print(f"Type Errors: {sample['errors']['type']}")
    print(f"Span Errors: {sample['errors']['span']}")
    print(f"Missing Errors: {sample['errors']['missing']}")
    print(f"Spurious Errors: {sample['errors']['spurious']}")
    print(f"Correct: {sample['correct']}")
    print("-" * 50)
# save_jsonl(data, "./datasets/aug/ai/deepseek-chat_zero_shot_ICL_errors.jsonl")

## 测试实验结果

In [2]:
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./output/aug/ai/refiner/Qwen2.5-72B-pos100-neg0-refiner-all-new.json")
counts = Counter()
for sent in data:
    if "refine_response" not in sent:
        continue
    sent['refine_response'] = sent['refine_response'].replace("Output: ", "")
    pred = extract_entities(sent['refine_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7022673031026253, 'recall': 0.7397862979258328, 'f1': 0.7205387205387206}


In [12]:
# output/aug/ai/modifier/deepseek-chat-pos100-neg10-modifier-all-100.json

import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

def get_ner_scores(eva_data, col = 'response'):
    counts = Counter()
    for sent in eva_data:
        if col not in sent:
            continue
        pred = extract_entities(sent[col])
        ref = sent['ners']
        counts = evaluate_sent(ref, pred, counts)
    scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
    return scores_ner

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./output/aug/ai/modifier/deepseek-chat-pos100-neg10-modifier-all-100.json")
for i in range(len(data)):
    data[i]['modifier_response'] = data[i]['modifier_response'].replace("**Final Annotated Sample**:\n", "")
# only keep the data that response == refine_response
# classify them to two parts: consist and non-consist
non_consist_data = [sent for sent in data if sent['response'] != sent['modifier_response']]
consis_data = [sent for sent in data if sent['response'] == sent['modifier_response']]


print("A1 performance: ",get_ner_scores(data, col = 'response'))
print("A2 performance: ", get_ner_scores(data, col = 'modifier_response'))
print("==========================")
print(len(consis_data), len(non_consist_data))
# get the consist data from data_2
print(get_ner_scores(consis_data, col = 'response'))
print(get_ner_scores(consis_data, col = 'modifier_response'))
print(get_ner_scores(non_consist_data, col = 'response'))
print(get_ner_scores(non_consist_data, col = 'modifier_response'))


A1 performance:  {'precision': 0.7476133651551312, 'recall': 0.7875549968573224, 'f1': 0.7670645852464034}
A2 performance:  {'precision': 0.7049689440993789, 'recall': 0.7133878064110623, 'f1': 0.7091533895657607}
191 159
{'precision': 0.8249118683901293, 'recall': 0.8407185628742515, 'f1': 0.8327402135231317}
{'precision': 0.8249118683901293, 'recall': 0.8407185628742515, 'f1': 0.8327402135231317}
{'precision': 0.6678787878787878, 'recall': 0.7288359788359788, 'f1': 0.6970271979759646}
{'precision': 0.5704874835309618, 'recall': 0.5727513227513228, 'f1': 0.5716171617161715}


In [14]:
# save non_consist_data
save_json(non_consist_data, "./output/aug/ai/deepseek-chat-pos100-neg10-modifier-all-100-non-consist.json")

In [7]:
# output/aug/ai/deepseek-chat-pos100-neg10-modifier-all-100-non-consist.json
data = load_json("./output/aug/ai/deepseek-chat-pos100-neg10-modifier-all-100-non-consist.json")
# get the ner scores
print(get_ner_scores(data, col = 'modifier_response'), len(data))
data = load_json("./output/aug/ai/refiner/deepseek-chat-pos100-neg0-modifier-noconsistent-refiner-all-new.json")
# get the ner scores
print(get_ner_scores(data, col = 'refine_response'), len(data))

{'precision': 0.5704874835309618, 'recall': 0.5727513227513228, 'f1': 0.5716171617161715} 159
{'precision': 0.6622114216281896, 'recall': 0.7208994708994709, 'f1': 0.6903103229892338} 159


In [11]:
counts = Counter()
print(len(data), len(consis_data))
for sent in data:
    pred = extract_entities(sent['refine_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
print(compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"]))
for sent in consis_data:
    pred = extract_entities(sent['modifier_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# return scores_ner
print(scores_ner)

159 191
{'precision': 0.6622114216281896, 'recall': 0.7208994708994709, 'f1': 0.6903103229892338}
{'precision': 0.7449223416965353, 'recall': 0.7837837837837838, 'f1': 0.7638591117917304}


In [ ]:
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType


# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./output/aug/ai/refiner/deepseek-chat-pos100-neg0-refiner-all-new.json")

# only keep the data that response == refine_response
# classify them to two parts: consist and non-consist
non_consist_data = [sent for sent in data if sent['response'] != sent['refine_response']]
consis_data = [sent for sent in data if sent['response'] == sent['refine_response']]

In [1]:
# output/aug/ai/modifier/deepseek-chat-pos100-neg10-modifier-all-100.json

import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

def get_ner_scores(eva_data, col = 'response'):
    counts = Counter()
    for sent in eva_data:
        if col not in sent:
            continue
        pred = extract_entities(sent[col])
        ref = sent['ners']
        counts = evaluate_sent(ref, pred, counts)
    scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
    return scores_ner

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./output/aug/ai/refiner/deepseek-chat-pos100-neg0-refiner-all-new.json")
for i in range(len(data)):
    data[i]['refine_response'] = data[i]['refine_response'].replace("Output: ", "")
# only keep the data that response == refine_response
# classify them to two parts: consist and non-consist
non_consist_data = [sent for sent in data if sent['response'] != sent['refine_response']]
consis_data = [sent for sent in data if sent['response'] == sent['refine_response']]

print(len(consis_data), len(non_consist_data))
# get the consist data from data_2
print(get_ner_scores(consis_data, col = 'response'))
print(get_ner_scores(consis_data, col = 'refine_response'))
print(get_ner_scores(non_consist_data, col = 'response'))
print(get_ner_scores(non_consist_data, col = 'refine_response'))

193 157
{'precision': 0.7868303571428571, 'recall': 0.822637106184364, 'f1': 0.8043354249857388}
{'precision': 0.7868303571428571, 'recall': 0.822637106184364, 'f1': 0.8043354249857388}
{'precision': 0.6728624535315985, 'recall': 0.7397820163487738, 'f1': 0.7047371836469825}
{'precision': 0.644927536231884, 'recall': 0.7275204359673024, 'f1': 0.6837387964148527}


In [21]:
def get_ner_scores(eva_data, col = 'response'):
    counts = Counter()
    for sent in eva_data:
        if col not in sent:
            continue
        pred = extract_entities(sent[col])
        ref = sent['ners']
        counts = evaluate_sent(ref, pred, counts)
    scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
    return scores_ner

In [27]:
# output/aug/ai/deepseek-chat-pos100-neg0-ann-all-new.json
data_1 = load_json("./output/aug/ai/deepseek-chat-demo100-ann-dev.json")
data_2 = load_json("./output/aug/ai/deepseek-chat-demo100-ann-dev-logits-logits2.json")
# get the consist data from data_1 and data_2, check whether response from data_1 == response from data_2
equal_data = []
unequal_data = []
for s_1, s_2 in zip(data_1, data_2):
    if s_1['response'] != s_2['response']:
        unequal_data.append(s_1)
    else:
        equal_data.append(s_1)
# print(len(equal_data), len(unequal_data))
print("A1 performance: ",get_ner_scores(data_1, col = 'response'))
print(len(equal_data), len(unequal_data))
# get the consist data from data_2
print(get_ner_scores(equal_data, col = 'response'))
print(get_ner_scores(unequal_data, col = 'response'))

A1 performance:  {'precision': 0.7400497512437811, 'recall': 0.7682375726275016, 'f1': 0.753880266075388}
210 140
{'precision': 0.812989921612542, 'recall': 0.8240635641316686, 'f1': 0.818489289740699}
{'precision': 0.6489510489510489, 'recall': 0.6946107784431138, 'f1': 0.6710050614605929}


In [5]:
# save the unequal data to json
save_json(unequal_data, "./output/aug/ai/deepseek-chat-pos100-neg0-ann-all-new-unequal.json")

In [11]:
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./output/aug/ai/refiner/deepseek-chat-pos100-neg0-unequal-refiner-all-new.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['refine_response'] = sent['refine_response'].replace("Output: ", "")
    pred = extract_entities(sent['refine_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6988265971316818, 'recall': 0.7454798331015299, 'f1': 0.7213997308209958}


In [15]:
from aug_prompt import TYPE_ERROR_PROMPT

import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./datasets/aug/ai/deepseek-chat_zero_shot_ICL_errors.json")

In [7]:
print(sent['response'])
print(sent['errors'])

<entity type="product">GATE</entity> includes an information extraction system called <entity type="product">ANNIE</entity> ( A Nearly-New Information Extraction System ) which is a set of modules comprising a <entity type="miscellaneous">tokenizer</entity> , a <entity type="miscellaneous">gazetteer</entity> , a <entity type="miscellaneous">sentence splitter</entity> , a <entity type="miscellaneous">Part-of-speech tagging</entity> , a <entity type="task">Named entity recognition</entity> transducer and a <entity type="miscellaneous">coreference tagger</entity> .
{'type': [['Part-of-speech tagging', 'miscellaneous', 'Part-of-speech tagging', 'task'], ['coreference tagger', 'miscellaneous', 'coreference tagger', 'product']], 'span': [['Named entity recognition', 'task', 'Named entity recognition transducer', 'product']], 'missing': [['A Nearly-New Information Extraction System', 'product'], ['information extraction', 'task']], 'spurious': []}


In [13]:
ent_format = """<entity type="{span_type}">{span_text}</entity>"""
# output_text = ""
if sent['errors']['type'] == []:
    # print("No span errors found.")
    output_text = "No span errors found."
else:
    ent_str_list = []
    for error in sent['errors']['type']:
        # each one build text like - "<entity type="annotated_type">text</entity>" may have wrong type. It is more likely "[suggested_type]".
        ent_str = "- "+ ent_format.format(span_type=error[1], span_text=error[0]) + " may have wrong type. It is more likely " + ent_format.format(span_type=error[3], span_text=error[2]) + "."
        ent_str_list.append(ent_str)
    output_text = "\n".join(ent_str_list)

In [14]:
print(output_text)

- <entity type="miscellaneous">Part-of-speech tagging</entity> may have wrong type. It is more likely <entity type="task">Part-of-speech tagging</entity>.
- <entity type="miscellaneous">coreference tagger</entity> may have wrong type. It is more likely <entity type="product">coreference tagger</entity>.


In [21]:
import json
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./datasets/aug/ai/deepseek-chat_zero_shot_ICL.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6505050505050505, 'recall': 0.6571428571428571, 'f1': 0.6538071065989847}


In [26]:
print(data[-1]['response'])
print( extract_entities(data[-1]['response']))

But <entity type="algorithm">perceptron</entity> models were made very unpopular by the book <entity type="product">Perceptrons</entity> by <entity type="researcher">Marvin Minsky</entity> and <entity type="researcher">Seymour Papert</entity> , published in 1969 .
[['perceptron', 'algorithm'], ['Perceptrons', 'product'], ['Marvin Minsky', 'researcher'], ['Seymour Papert', 'researcher']]


In [22]:
data_samples = []
for i in range(len(data)):
    data[i]['response'] = data[i]['response'].replace("Output: ", "")
    data[i]['pred_ners'] = extract_entities(data[i]['response'])
    data_samples.append(
        {
            "llm_prediction": data[i]['pred_ners'],
            "ground_truth": data[i]['ners'],
        }
    )
# 统计每个样本的错误类型
# --- 2. Run Analysis ---
total_results: CounterType[str] = collections.Counter() # Aggregates counts across all samples
total_gt_entities: int = 0
total_llm_entities: int = 0

print(f"Starting NER error analysis on {len(data_samples)} samples...")

for i, sample in enumerate(data_samples):
    if 'llm_prediction' not in sample or 'ground_truth' not in sample:
        print(f"Warning: Sample {i+1} is missing 'llm_prediction' or 'ground_truth'. Skipping.")
        continue

    llm_pred = sample['llm_prediction']
    gt = sample['ground_truth']

    # Analyze the single sample
    try:
        sample_results = analyze_ner_errors(llm_pred, gt)
        # Update total results
        total_results.update(sample_results)
        # Update total entity counts
        total_gt_entities += len(gt)
        total_llm_entities += len(llm_pred)
    except Exception as e:
        print(f"Error processing sample {i+1}: {e}")
        # Optionally add more details about the sample causing the error

print("Analysis complete.")


# --- 3. Display Results ---
print("\n--- Overall NER Error Analysis ---")
print(f"Total Samples Analyzed: {len(data_samples)}")
print("-" * 35)
print(f"Total Ground Truth Entities:   {total_gt_entities}")
print(f"Total LLM Predicted Entities:  {total_llm_entities}")
print("-" * 35)
print("Entity Counts:")
print(f"  Correct Matches:      {total_results['correct']}")
print(f"  Entity Type Errors:   {total_results['type']}")
print(f"  Entity Span Errors:   {total_results['span']} (using substring matching)")
print(f"  Missing Entities:     {total_results['missing']}")
print(f"  Spurious Entities:    {total_results['spurious']}")
print("-" * 35)

# --- Performance Metrics (Strict - Type/Span count as FP & FN) ---
# True Positives (TP) = Correct Matches
# False Positives (FP) = Spurious Errors + Type Errors + Span Errors
# False Negatives (FN) = Missing Errors + Type Errors + Span Errors

tp = total_results['correct']
fp = total_results['spurious'] + total_results['type'] + total_results['span']
fn = total_results['missing'] + total_results['type'] + total_results['span']

# Avoid division by zero
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Performance Metrics (Strict Matching):")
print(f"  Precision: {precision:.4f}  (TP / (TP + FP))")
print(f"  Recall:    {recall:.4f}  (TP / (TP + FN))")
print(f"  F1-Score:  {f1_score:.4f}")
print("-" * 35)

Starting NER error analysis on 100 samples...
Analysis complete.

--- Overall NER Error Analysis ---
Total Samples Analyzed: 100
-----------------------------------
Total Ground Truth Entities:   490
Total LLM Predicted Entities:  495
-----------------------------------
Entity Counts:
  Correct Matches:      321
  Entity Type Errors:   82
  Entity Span Errors:   51 (using substring matching)
  Missing Entities:     36
  Spurious Entities:    41
-----------------------------------
Performance Metrics (Strict Matching):
  Precision: 0.6485  (TP / (TP + FP))
  Recall:    0.6551  (TP / (TP + FN))
  F1-Score:  0.6518
-----------------------------------


### 验证100 shots实验的效果

In [9]:
import json
import os
from collections import Counter
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1


# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data = load_json("./datasets/aug/ai/deepseek-chat_50-ann4reflection.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['entities']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.75, 'recall': 0.7815126050420168, 'f1': 0.7654320987654322}


找出和总结错误类型

In [4]:
data[0]

{'text': 'GATE includes an information extraction system called ANNIE ( A Nearly-New Information Extraction System ) which is a set of modules comprising a tokenizer , a gazetteer , a sentence splitter , a Part-of-speech tagging , a Named entity recognition transducer and a coreference tagger .',
 'target': '<entity type="product">GATE</entity> includes an <entity type="task">information extraction</entity> system called <entity type="product">ANNIE</entity> ( <entity type="product">A Nearly-New Information Extraction System</entity> ) which is a set of modules comprising a <entity type="miscellaneous">tokenizer</entity> , a <entity type="miscellaneous">gazetteer</entity> , a <entity type="miscellaneous">sentence splitter</entity> , a <entity type="task">Part-of-speech tagging</entity> , a <entity type="product">Named entity recognition transducer</entity> and a <entity type="product">coreference tagger</entity> .',
 'entities': [['GATE', 'product'],
  ['information extraction', 'task'

In [10]:
errors = []
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['entities']
    # removing the "Output: " prefix from the response
    sent["response"] = sent["response"].replace("Output: ", "")
    if pred != ref:
        errors.append(sent)
print(len(errors))

29


In [11]:
# Save errors to a JSON file
output_path = "./datasets/aug/ai/negatives.json"
with open(output_path, "w") as f:
    json.dump(errors, f, indent=4)

**Ask LLMs to self-reflection to summarize the error types and generate new prompts**

In [6]:
text = ""
for error in errors:
    # text += "Text: " +  error["text"] + "\n"
    text += "Prediction: " + str(error["response"]) + "\n"
    text += "Reference: " + str(error["target"]) + "\n"
    text += "\n\n"
with open("./output/aug/ai/errors.txt", "w") as f:
    f.write(text)

In [7]:
errors[0]

{'text': 'GATE includes an information extraction system called ANNIE ( A Nearly-New Information Extraction System ) which is a set of modules comprising a tokenizer , a gazetteer , a sentence splitter , a Part-of-speech tagging , a Named entity recognition transducer and a coreference tagger .',
 'target': '<entity type="product">GATE</entity> includes an <entity type="task">information extraction</entity> system called <entity type="product">ANNIE</entity> ( <entity type="product">A Nearly-New Information Extraction System</entity> ) which is a set of modules comprising a <entity type="miscellaneous">tokenizer</entity> , a <entity type="miscellaneous">gazetteer</entity> , a <entity type="miscellaneous">sentence splitter</entity> , a <entity type="task">Part-of-speech tagging</entity> , a <entity type="product">Named entity recognition transducer</entity> and a <entity type="product">coreference tagger</entity> .',
 'response': '<entity type="product">GATE</entity> includes an <entity

In [20]:
text = ""
for error in errors:
    # text += "Text: " +  error["text"] + "\n"
    # remove the "Output: " from the error['response]
    error['response'] = error['response'].replace("Output: ", "")
    text += "Prediction: " + str(error["response"]) + "\n"
    text += "Reference: " + str(error["target"]) + "\n"
    text += "\n\n"
with open("./output/aug/ai/errors.txt", "w") as f:
    f.write(text)

Error STAT
- Type Error: +, 

In [12]:
type_errors = {0, }
boundary_erors = {1, }

In [15]:
errors[1]

{'text': 'Other ways anomalous propagation is recorded is by troposcatter s causing irregularities in the troposphere , scattering due to meteor s , refraction in the ionized regions and layers of the ionosphere , and reflection from the ionosphere .',
 'target': 'Other ways <entity type="miscellaneous">anomalous propagation</entity> is recorded is by <entity type="miscellaneous">troposcatter</entity> s causing irregularities in the <entity type="miscellaneous">troposphere</entity> , scattering due to meteor s , refraction in the <entity type="miscellaneous">ionized regions</entity> and layers of the <entity type="miscellaneous">ionosphere</entity> , and reflection from the <entity type="miscellaneous">ionosphere</entity> .',
 'response': 'Output: Other ways <entity type="task">anomalous propagation</entity> is recorded is by <entity type="miscellaneous">troposcatter</entity> s causing irregularities in the <entity type="miscellaneous">troposphere</entity> , scattering due to <entity t

## 实验结果

In [1]:
import json
import os
from collections import Counter
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1
import random

In [18]:
data = load_json(f"./output/aug/ai/deepseek-chat_R1.json")
# randomly sample 100 samles
# random.seed(42)
# data = random.sample(data, 100)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7406305770374777, 'recall': 0.7825267127592709, 'f1': 0.7610024449877749}


In [4]:
data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg10-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7213586643638457, 'recall': 0.7875549968573224, 'f1': 0.7530048076923076}


In [5]:
data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg0-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7176403207331042, 'recall': 0.7875549968573224, 'f1': 0.7509739286784537}


In [25]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-False.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg0-ann-all-no-guideline-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7275922671353251, 'recall': 0.7806411062225016, 'f1': 0.7531837477258945}


In [26]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-False.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg0-ann-all-no-guideline-retrive-False.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.739001189060642, 'recall': 0.781269641734758, 'f1': 0.7595478154598228}


In [ ]:
# 有guideline的
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg0-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.727751756440281, 'recall': 0.781269641734758, 'f1': 0.7535616853591999}


In [33]:
# 

# 有guideline的
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg0-ann-all-no-guideline-retrive-True-2.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7248831775700935, 'recall': 0.7800125707102451, 'f1': 0.7514380865879504}


In [20]:
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg10-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7476133651551312, 'recall': 0.7875549968573224, 'f1': 0.7670645852464034}


In [21]:
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg20-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7393867924528302, 'recall': 0.7881835323695788, 'f1': 0.7630057803468209}


In [29]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-False.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-False.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6893424036281179, 'recall': 0.764299182903834, 'f1': 0.7248882265275707}


In [13]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-False.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7062857142857143, 'recall': 0.7768698931489629, 'f1': 0.7398982340616581}


In [ ]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-False.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg0-ann-all-no-guideline-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7275922671353251, 'recall': 0.7806411062225016, 'f1': 0.7531837477258945}


In [8]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all.json

data = load_json(f"./output/ai/deepseek-chat_50.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['entities']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7289416846652268, 'recall': 0.7876312718786465, 'f1': 0.7571508693213684}


In [30]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg0-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7176403207331042, 'recall': 0.7875549968573224, 'f1': 0.7509739286784537}


In [31]:
# output/aug/ai/deepseek-chat-pos50-neg0-ann-all.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos50-neg10-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7213586643638457, 'recall': 0.7875549968573224, 'f1': 0.7530048076923076}


In [6]:
# output/aug/ai/deepseek-chat-pos100-neg10-ann-all-no-guideline-retrive-True.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg10-ann-all-no-guideline-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7263652378156195, 'recall': 0.7774984286612193, 'f1': 0.7510625379477838}


In [7]:
# output/aug/ai/deepseek-chat-pos100-neg10-ann-all-no-guideline-retrive-True.json

data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg20-ann-all-no-guideline-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7359050445103857, 'recall': 0.7793840351979887, 'f1': 0.7570207570207571}


**Qwen**

In [2]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6806039488966318, 'recall': 0.7366436203645506, 'f1': 0.7075158466646544}


In [3]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos50-neg10-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7003588516746412, 'recall': 0.7360150848522942, 'f1': 0.717744406987435}


In [8]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos50-neg20-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7115268557634279, 'recall': 0.7410433689503457, 'f1': 0.7259852216748769}


In [3]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos100-neg0-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7017334130304842, 'recall': 0.7379006913890634, 'f1': 0.7193627450980393}


In [4]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos100-neg10-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.711178247734139, 'recall': 0.7397862979258328, 'f1': 0.7252002464571781}


In [5]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos100-neg20-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.703125, 'recall': 0.7353865493400377, 'f1': 0.71889400921659}


In [6]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all-no-definition-retrive-False.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6860934795152914, 'recall': 0.7473287240729101, 'f1': 0.7154031287605295}


In [7]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all-no-definition-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.688533941814033, 'recall': 0.7586423632935261, 'f1': 0.7218899521531101}


In [8]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos100-neg0-ann-all-no-definition-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6895137668424136, 'recall': 0.7397862979258328, 'f1': 0.7137659187386296}


In [9]:
# output/aug/ai/Qwen2.5-72B-pos50-neg0-ann-all.json
data = load_json(f"./output/aug/ai/Qwen2.5-72B-pos100-neg10-ann-all-no-definition-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.6971125515615793, 'recall': 0.7435575109993715, 'f1': 0.7195863746958637}


In [7]:
# output/aug/ai/deepseek-chat-pos100-neg10-ann-all-retrive-True.json
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg20-ann-all-no-guideline-retrive-True.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7359050445103857, 'recall': 0.7793840351979887, 'f1': 0.7570207570207571}


In [8]:
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg10-ann-all.json")
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7476133651551312, 'recall': 0.7875549968573224, 'f1': 0.7670645852464034}


In [2]:
data = load_json(f"./output/aug/ai/modifier/deepseek-chat-pos100-neg10-modifier-all.json")
counts = Counter()
for sent in data:
    if "modifier_response" not in sent:
        continue
    pred = extract_entities(sent['modifier_response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7094763092269327, 'recall': 0.7152734129478315, 'f1': 0.7123630672926449}
